In [ ]:
# ==============================================================================
# 📓 Advanced RAG Cookbook: 03_embeddings_and_vectordb.ipynb
# ==============================================================================
# الهدف: فهم قواعد البيانات الاتجاهية (Vector DBs)، تطبيق ChromaDB و Qdrant،
# وفهم كيفية بناء Ingestion & Retrieval Pipelines للإنتاج الإنتاجي للشركات.
# ==============================================================================

# تثبيت المكتبات المطلوبة للتشغيل
# !pip install langchain-community langchain-chroma qdrant-client sentence-transformers pypdf

import os
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.vectorstores import Qdrant
from qdrant_client import QdrantClient

print("✅ تم استيراد كل المكتبات بنجاح!")

<div dir="rtl">

## 1. ما هي قواعد البيانات الاتجاهية (Vector Databases)؟

### 📌 الشرح النظري:
قواعد البيانات التقليدية (SQL / NoSQL) تبحث عن المطابقة الحرفية للكلمات (`EXACT MATCH`). أما **Vector DBs** فتخزن النصوص بعد تحويلها لمكافئات رياضية (`Vectors/Embeddings`)، وتبحث بناءً على **المعنى والدلالة (`Semantic Similarity`)**.

---

### 📊 مقارنة أشهر قواعد البيانات الاتجاهية في سوق العمل:

| قاعدة البيانات | نوع الاستخدام (Use Case) | نمط التشغيل (Deployment) | أداء التصفية (Filtering) |
| :--- | :--- | :--- | :--- |
| **ChromaDB** | التطوير السريع والمشاريع الصغيرة (PoC) | محلي (In-Memory / File) | متوسط |
| **Qdrant** | الأنظمة الإنتاجية الضخمة (Production-Grade) | محلي (Docker) أو سحابي (Cloud) | فائق السرعة والدقة |
| **Pinecone** | التطبيقات السحابية المدارة بالكامل | سحابي فقط (Fully Managed) | ممتاز وسهل |
| **PGVector** | للشركات التي تعتمد بالفعل على PostgreSQL | ميزة إضافية على Postgres | ممتاز للمشاريع الهجينة |

</div>

<div dir="rtl">

## 2. التطبيق المحلي الأول: ChromaDB

قاعدة بيانات خفيفة ومناسبة للتجارب السريعة بدون إعداد سيرفرات.

</div>

In [ ]:
# ==============================================================================
# 2. ChromaDB Demo
# ==============================================================================

# 1. إعداد نموذج الـ Embeddings
embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 2. تجهيز بيانات تجريبية مع Metadata
sample_docs = [
    Document(
        page_content="FastAPI is a modern web framework for building APIs with Python.",
        metadata={"category": "Backend", "author": "Tiangolo"}
    ),
    Document(
        page_content="Qdrant and ChromaDB are specialized vector databases for similarity search.",
        metadata={"category": "Database", "author": "RAG Team"}
    ),
    Document(
        page_content="Deep Learning and Neural Networks drive modern AI capabilities.",
        metadata={"category": "AI", "author": "Andrew Ng"}
    )
]

# 3. إنشاؤها وتخزين البيانات في ChromaDB محلياً على القرص
chroma_db = Chroma.from_documents(
    documents=sample_docs,
    embedding=embeddings_model,
    persist_directory="./chroma_db_data"
)

# 4. البحث التشابهي (Similarity Search)
query = "What tools are used for vector storage?"
results = chroma_db.similarity_search(query, k=1)

print("--- ChromaDB Search Result ---")
print(f"🔍 Query: {query}")
print(f"📄 Content: {results[0].page_content}")
print(f"🏷️ Metadata: {results[0].metadata}\n")

<div dir="rtl">

## 3. التطبيق العملي الثاني: Qdrant

قاعدة بيانات احترافية مكتوبة بلغة **Rust**، تُستخدم في بيئات الإنتاج الفعلية للشركات بفضل سرعتها العالية ودعمها للـ Payload Filtering.

</div>

In [ ]:
# ==============================================================================
# 3. Qdrant Demo (In-Memory with Filtering)
# ==============================================================================

# 1. إنشاء واستيراد البيانات داخل Qdrant في الذاكرة
qdrant_db = Qdrant.from_documents(
    documents=sample_docs,
    embedding=embeddings_model,
    location=":memory:",
    collection_name="rag_cookbook"
)

# 2. البحث التشابهي مع تصفية Metadata (Filtering)
query = "Tell me about modern Python tools"
filtered_results = qdrant_db.similarity_search(
    query, 
    k=1, 
    filter={"category": "Backend"}
)

print("--- Qdrant Search Result with Filtering ---")
print(f"🔍 Query: {query}")
print(f"📄 Content: {filtered_results[0].page_content}")
print(f"🏷️ Metadata: {filtered_results[0].metadata}\n")

<div dir="rtl">

## 4. كيف تعمل أنظمة الـ RAG داخل الشركات؟ (Production Architecture)

في الشركات، ينقسم الكود دائماً إلى خطين منفصلين:

1. **Ingestion Pipeline (`ingest.py`):** خط يعمل دورياً أو مرة واحدة، يقوم بـ Loading -> Chunking -> Embedding -> ثم رفع النتائج إلى Cloud Vector DB.
2. **Retrieval Pipeline (`app.py`):** سيرفر الـ Backend الذي يستقبل أسئلة المستخدمين، يبحث أونلاين في الـ Vector DB مباشرةً دون إعادة تقطيع البيانات.

</div>

In [ ]:
# ==============================================================================
# 4. Production Pipeline Part 1: Ingestion Pipeline
# ==============================================================================

# محاكاة مستند طويل من الشركة
raw_company_text = """
Company Policy Document 2026.
Employees are entitled to 21 days of paid annual leave after completing probation.
Remote work is allowed up to 2 days per week with manager approval.
Health insurance coverage includes dental and optical care up to $5,000 annually.
"""

# 1. Chunking
text_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)
company_chunks = text_splitter.split_text(raw_company_text)

# 2. Upload to Cloud / Local Qdrant Server
# ملاحظة: للاتصال بالشركة أونلاين استبدل location بـ url="https://..." و api_key="..."
qdrant_production = Qdrant.from_texts(
    texts=company_chunks,
    embedding=embeddings_model,
    location=":memory:",  # في الشركة نضع رابط السحابة هنا
    collection_name="production_company_docs"
)

print("✅ [Ingestion Success] تم تقطيع بيانات الشركة وتحويلها لـ Vectors وحفظها سحابياً!")

In [ ]:
# ==============================================================================
# 5. Production Pipeline Part 2: Retrieval Pipeline
# ==============================================================================

# في تطبيق الـ Backend، نحن لا نُعيد التقطيع، بل نتصل بقاعدة البيانات مباشرةً
user_question = "How many remote working days are allowed?"

# الاستعلام السريع مباشرة من البيانات المرفوعة
retrieved_docs = qdrant_production.similarity_search(user_question, k=2)

print(f"❓ سؤال المستخدم: {user_question}\n")
print("📌 النتائج المسترجعة مباشرة من الـ Vector DB:")
for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

<div dir="rtl">

## 📝 الخلاصة والتوصيات للـ Notebook

1. **الفرق بين المطور المحلي والشركة:** التقطيع والتحويل خطوات إجبارية في الحالتين، ولكن في الشركات نرفع المتجهات أونلاين لـ Cloud Vector DB (مثل Qdrant Cloud أو Pinecone).
2. **فصل المسئوليات:** ملف الـ Ingestion يعمل خلف الكواليس لتنظيم البيانات، بينما ملف الـ App يجيب على المستخدمين بسرعة من خلال الـ Search المباشر.
3. **الاختيار المناسب:** ChromaDB ممتاز للتجارب والـ PoC، و Qdrant هو الخيار الأقوى للأنظمة الإنتاجية (Production RAG).

</div>